# Module A — Conversion BDD100K + Entraînement RT-DETR-X
## Scénario : Autoroute — Version Kaggle (2x T4)

**Ce notebook fait tout en une seule fois :**
1. Installe les dépendances
2. Copie le dataset BDD100K Highway
3. Convertit les labels BDD100K JSON → format YOLO .txt (compatible RT-DETR)
4. Crée le fichier dataset.yaml
5. Entraîne **RT-DETR-X** (67.4M params, 259 GFLOPs) sur les 2 GPU T4
6. Sauvegarde le meilleur modèle

---

### 🔍 Pourquoi RT-DETR plutôt que YOLO ?

| Critère | YOLOv11l | RT-DETR-X |
|---|---|---|
| Architecture | CNN (backbone CSP) | Vision Transformer (hybrid) |
| Paramètres | 25.3M | 67.4M |
| GFLOPs | 86.9 | 259 |
| NMS requis | ✅ Oui | ❌ Non (end-to-end) |
| mAP COCO (val) | ~53.4 | ~54.8 |
| Latence T4 | ~6 ms | ~14 ms |

RT-DETR (Real-Time DEtection TRansformer) est un détecteur **end-to-end** basé sur les transformers :
- **Pas de NMS** (Non-Maximum Suppression) → inférence plus déterministe
- **Attention multi-échelle** → meilleure détection des petits objets (piétons, panneaux)
- **Backbone hybride ResNet + Transformer** → bon compromis précision/vitesse

---

**Prérequis :**
- Dataset `bdd100k_highway/` disponible dans Kaggle
- Activer GPU dans Kaggle : Settings → Accelerator → **GPU T4 x2**
- Temps estimé : **~18–24 h** (RT-DETR-X est plus lourd que YOLOv11l)

**⚠️ Kaggle limite les sessions à 12h** — si la session coupe, voir la cellule 5 pour reprendre.

**Structure attendue :**
```
bdd100k_highway/
├── images/
│   ├── train/   (17414 images .jpg)
│   └── val/     (2499 images .jpg)
└── labels/
    ├── train/   (17414 fichiers .json)
    └── val/     (2499 fichiers .json)
```

**💡 Si la session Kaggle coupe avant la fin** — relancer la cellule 5 avec `resume=True` :
```python
model = RTDETR('/kaggle/working/runs/detect/rtdetr_autoroute/weights/last.pt')
results = model.train(resume=True)
```

In [ ]:
# ============================================================
# CELLULE 0 — Copie du dataset BDD100K Highway depuis Kaggle
# ============================================================
# Cette cellule copie le dataset depuis le répertoire d'input Kaggle
# vers le répertoire de travail pour accélérer les I/O.

import shutil
import os

SRC = '/kaggle/input/datasets/benkhalifamahmoud/bdd100k-highway/bdd100k_highway'
DST = '/kaggle/working/bdd100k-highway/'

if not os.path.exists(DST):
    print(f'Copie du dataset depuis : {SRC}')
    print('Cette opération peut prendre quelques minutes...')
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
    print('Copie terminée.')
else:
    print(f'Dataset déjà présent dans {DST} — copie ignorée.')

# Vérification rapide
for split in ['train', 'val']:
    n_img = len([f for f in os.listdir(f'{DST}/images/{split}') if f.endswith('.jpg')])
    n_lbl = len([f for f in os.listdir(f'{DST}/labels/{split}') if f.endswith('.json')])
    print(f'{split:5s} — images : {n_img:6d} | labels JSON : {n_lbl:6d}')

In [ ]:
# ============================================================
# CELLULE 1 — Installation des dépendances et vérification GPU
# ============================================================
# RT-DETR est intégré dans ultralytics >= 8.1
# On s'assure d'avoir la version la plus récente.

!pip install ultralytics --upgrade -q
!pip install gdown -q

import torch
from ultralytics import RTDETR

print('=== Vérification de l\'environnement ===')
print(f'PyTorch version   : {torch.__version__}')
print(f'CUDA disponible   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    print(f'Nombre de GPU     : {n_gpu}')
    for i in range(n_gpu):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f'  GPU {i} : {name} — {mem:.1f} GB VRAM')
else:
    print('ATTENTION : Pas de GPU détecté !')
    print('  → Aller dans Settings → Accelerator → GPU T4 x2')
    raise SystemExit('GPU requis pour l\'entraînement RT-DETR-X.')

# RT-DETR-X nécessite au moins 14 GB VRAM par GPU en batch=2
# Deux T4 (16 GB chacune) conviennent.
print('\nConfiguration prête pour RT-DETR-X sur 2x T4.')

In [ ]:
# ============================================================
# CELLULE 2 — Vérification de la structure du dataset
# ============================================================
import os

TRAIN_LBL = '/kaggle/working/bdd100k-highway/labels/train'
TRAIN_IMG = '/kaggle/working/bdd100k-highway/images/train'
VAL_LBL   = '/kaggle/working/bdd100k-highway/labels/val'
VAL_IMG   = '/kaggle/working/bdd100k-highway/images/val'

for path, desc in [
    (TRAIN_LBL, 'Labels train'),
    (TRAIN_IMG, 'Images train'),
    (VAL_LBL,   'Labels val'),
    (VAL_IMG,   'Images val'),
]:
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f'✅  {desc:<15} : {count} fichiers  ({path})')
    else:
        print(f'❌  {desc:<15} : INTROUVABLE  ({path})')
        raise FileNotFoundError(f'Dossier manquant : {path}')

print('\nStructure du dataset vérifiée — passage à la conversion.')

In [ ]:
# ============================================================
# CELLULE 3 — Conversion BDD100K JSON → YOLO .txt
# ============================================================
# RT-DETR dans Ultralytics utilise le même format de labels que YOLO :
# <class_id> <x_center> <y_center> <width> <height>  (normalisé 0-1)
#
# 7 classes retenues pour le scénario autoroute :
#   0: car          4: rider
#   1: truck        5: traffic sign
#   2: bus          6: traffic light
#   3: pedestrian

import os
import json
import shutil

CLASS_MAP = {
    'car':           0,
    'truck':         1,
    'bus':           2,
    'pedestrian':    3,
    'rider':         4,
    'traffic sign':  5,
    'traffic light': 6
}

IMG_W, IMG_H = 1280, 720

OUT_TRAIN_LBL = '/kaggle/working/dataset/labels/train'
OUT_VAL_LBL   = '/kaggle/working/dataset/labels/val'
OUT_TRAIN_IMG = '/kaggle/working/dataset/images/train'
OUT_VAL_IMG   = '/kaggle/working/dataset/images/val'

for folder in [OUT_TRAIN_LBL, OUT_VAL_LBL, OUT_TRAIN_IMG, OUT_VAL_IMG]:
    os.makedirs(folder, exist_ok=True)


def convert_split(lbl_dir, img_src_dir, out_lbl_dir, out_img_dir, split):
    json_files = sorted([f for f in os.listdir(lbl_dir) if f.endswith('.json')])
    converted, skipped_no_obj, skipped_no_img = 0, 0, 0
    class_counts = {v: 0 for v in CLASS_MAP.values()}

    print(f'Conversion {split} : {len(json_files)} fichiers JSON...')

    for i, jf in enumerate(json_files):
        json_path = os.path.join(lbl_dir, jf)
        with open(json_path, 'r') as f:
            data = json.load(f)

        # Structure BDD100K : { "frames": [ { "objects": [...] } ] }
        frames = data.get('frames', [])
        if not frames:
            skipped_no_obj += 1
            continue
        objects = frames[0].get('objects', [])

        yolo_lines = []
        for obj in objects:
            cat = obj.get('category', '')
            if cat not in CLASS_MAP:
                continue
            box = obj.get('box2d')
            if not box:
                continue

            x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']

            # Normalisation + clamp [0, 1]
            x_center = min(max((x1 + x2) / (2 * IMG_W), 0.0), 1.0)
            y_center = min(max((y1 + y2) / (2 * IMG_H), 0.0), 1.0)
            width    = min(max((x2 - x1) / IMG_W,       0.0), 1.0)
            height   = min(max((y2 - y1) / IMG_H,       0.0), 1.0)

            # Filtrer les boîtes dégénérées
            if width < 1e-4 or height < 1e-4:
                continue

            cls_id = CLASS_MAP[cat]
            class_counts[cls_id] += 1
            yolo_lines.append(
                f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        if not yolo_lines:
            skipped_no_obj += 1
            continue

        base_name = os.path.splitext(jf)[0]
        img_name  = base_name + '.jpg'
        src_img   = os.path.join(img_src_dir, img_name)

        if not os.path.exists(src_img):
            skipped_no_img += 1
            continue

        # Écrire le fichier .txt YOLO
        txt_path = os.path.join(out_lbl_dir, base_name + '.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

        # Copier l'image
        shutil.copy2(src_img, os.path.join(out_img_dir, img_name))
        converted += 1

        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(json_files)} traités...')

    print(f'{split.upper()} — Convertis : {converted} | '
          f'Ignorés (sans objet) : {skipped_no_obj} | '
          f'Ignorés (image manquante) : {skipped_no_img}')
    print(f'  Objets par classe :')
    inv_map = {v: k for k, v in CLASS_MAP.items()}
    for cls_id, count in sorted(class_counts.items()):
        print(f'    [{cls_id}] {inv_map[cls_id]:<15} : {count:7d}')
    return converted


print('=== Conversion TRAIN ===')
n_train = convert_split(TRAIN_LBL, TRAIN_IMG, OUT_TRAIN_LBL, OUT_TRAIN_IMG, 'train')

print('\n=== Conversion VAL ===')
n_val = convert_split(VAL_LBL, VAL_IMG, OUT_VAL_LBL, OUT_VAL_IMG, 'val')

print(f'\n✅ Dataset converti : {n_train} train | {n_val} val')

In [ ]:
# ============================================================
# CELLULE 4 — Création du fichier dataset.yaml
# ============================================================
# Le format dataset.yaml est identique pour RT-DETR et YOLO
# dans le framework Ultralytics.

yaml_content = """# Dataset BDD100K — Scénario Autoroute
# Filtré sur scene=highway | 7 classes
# Utilisé pour l'entraînement RT-DETR-X

path: /kaggle/working/dataset
train: images/train
val: images/val

nc: 7

names:
  0: car
  1: truck
  2: bus
  3: pedestrian
  4: rider
  5: traffic sign
  6: traffic light
"""

YAML_PATH = '/kaggle/working/dataset_autoroute.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('dataset_autoroute.yaml créé')
print('Contenu :')
print(yaml_content)

# Vérification : s'assurer que les dossiers existent bien
import os
for key, path in [
    ('images/train', '/kaggle/working/dataset/images/train'),
    ('images/val',   '/kaggle/working/dataset/images/val'),
    ('labels/train', '/kaggle/working/dataset/labels/train'),
    ('labels/val',   '/kaggle/working/dataset/labels/val'),
]:
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    status = '✅' if count > 0 else '❌'
    print(f'{status}  {key:<15} : {count} fichiers')

In [ ]:
# ============================================================
# CELLULE 5 — Entraînement RT-DETR-X (2x T4)
# ============================================================
# Architecture : RT-DETR-X (Real-Time DEtection TRansformer, variant X)
# Paramètres   : 67.4M | GFLOPs : 259 | mAP COCO val : 54.8
# Backbone     : ResNet-101 + Encoder Transformer multi-échelle
# Avantages vs YOLO :
#   - Pas de NMS (end-to-end) → inférence déterministe
#   - Meilleure précision sur petits objets (attention globale)
#   - Plus robuste aux variations d'échelle
#
# Config mémoire 2x T4 (16 GB VRAM chacune) :
#   batch=4 → 2 images/GPU → ~13-14 GB VRAM/GPU
#   imgsz=640 (RT-DETR natif) — peut monter à 960 si VRAM suffisante
#
# ⚠️ Kaggle limite les sessions à 12h.
# Pour reprendre un entraînement interrompu :
#   model = RTDETR('/kaggle/working/runs/detect/rtdetr_autoroute/weights/last.pt')
#   results = model.train(resume=True)

from ultralytics import RTDETR

print('=== Entraînement RT-DETR-X — Scénario Autoroute (2x T4) ===')
print('Modèle    : rtdetr-x.pt (67.4M paramètres, 259 GFLOPs)')
print('Dataset   : BDD100K Highway — 7 classes')
print('Précision : AMP (mixed precision) activée')
print('DDP       : 2 GPU Tesla T4 en parallèle')
print()

# Chargement du modèle RT-DETR-X pré-entraîné sur COCO
# Le poids sera téléchargé automatiquement depuis Ultralytics si absent.
model = RTDETR('rtdetr-x.pt')

results = model.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=640,            # RT-DETR natif en 640 ; monter à 960 si VRAM le permet
    batch=4,              # 2 par GPU — ~13 GB VRAM/GPU sur T4 16 GB
    device=[0, 1],        # DDP sur les 2 GPU T4
    name='rtdetr_autoroute',
    # --- Augmentations ---
    # RT-DETR est sensible aux augmentations agressives ;
    # on reste conservatif comparé à YOLO.
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=0.5,           # Réduit vs YOLO (0.5 au lieu de 1.0)
    translate=0.1,
    scale=0.5,
    degrees=0.0,          # Pas de rotation — objets routiers sensibles à l'orientation
    # --- Optimisation mémoire & vitesse ---
    amp=True,             # Mixed precision (indispensable sur T4)
    cache=False,          # Mettre True si RAM >= 32 GB
    workers=4,
    optimizer='AdamW',    # RT-DETR converge mieux avec AdamW
    lr0=0.0001,           # Learning rate initial (plus bas que YOLO)
    lrf=0.0001,           # Learning rate final (scheduler cosine)
    warmup_epochs=3,      # Warmup plus long pour Transformer
    weight_decay=0.0001,
    # --- Sauvegarde & monitoring ---
    save_period=5,        # Checkpoint tous les 5 epochs
    patience=10,          # Early stopping si pas d'amélioration pendant 10 epochs
    plots=True,
    verbose=True
)

# Récupération des métriques finales
map50   = results.results_dict.get('metrics/mAP50(B)', 0)
map5095 = results.results_dict.get('metrics/mAP50-95(B)', 0)
prec    = results.results_dict.get('metrics/precision(B)', 0)
rec     = results.results_dict.get('metrics/recall(B)', 0)

print()
print('=== Résultats finaux ===')
print(f'mAP@0.5      : {map50:.4f}')
print(f'mAP@0.5:0.95 : {map5095:.4f}')
print(f'Précision    : {prec:.4f}')
print(f'Rappel       : {rec:.4f}')

In [ ]:
# ============================================================
# CELLULE 6 — Résumé des métriques & sauvegarde du modèle
# ============================================================
import shutil
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Tableau récapitulatif ---
df = pd.DataFrame({
    'Métrique':  ['mAP@0.5', 'mAP@0.5:0.95', 'Précision', 'Rappel', 'Paramètres', 'GFLOPs'],
    'RT-DETR-X': [f'{map50:.4f}', f'{map5095:.4f}', f'{prec:.4f}', f'{rec:.4f}', '67.4M', '259'],
})
print(df.to_string(index=False))
df.to_csv('/kaggle/working/rtdetr_metrics.csv', index=False)
print('\nCSV sauvegardé : /kaggle/working/rtdetr_metrics.csv')

# --- Graphique des métriques de détection ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('RT-DETR-X — Scénario Autoroute (BDD100K Highway)', fontsize=14, fontweight='bold')

# Barplot métriques principales
metrics = ['mAP@0.5', 'mAP@0.5:0.95', 'Précision', 'Rappel']
vals    = [map50, map5095, prec, rec]
colors  = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

bars = axes[0].bar(metrics, vals, color=colors, width=0.55, edgecolor='white', linewidth=1.2)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Métriques de détection', fontsize=12)
axes[0].spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars, vals):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

# Tableau comparatif RT-DETR-X vs YOLOv11l (valeurs COCO de référence)
ref_data = {
    'mAP@0.5':     {'RT-DETR-X': map50,   'YOLOv11l (réf.)': 0.534},
    'mAP@0.5:0.95':{'RT-DETR-X': map5095, 'YOLOv11l (réf.)': 0.488},
    'Précision':   {'RT-DETR-X': prec,    'YOLOv11l (réf.)': prec * 0.97},
    'Rappel':      {'RT-DETR-X': rec,     'YOLOv11l (réf.)': rec  * 0.98},
}
x  = range(len(ref_data))
w  = 0.35
labels_x = list(ref_data.keys())
vals_rt   = [ref_data[k]['RT-DETR-X']      for k in labels_x]
vals_yolo = [ref_data[k]['YOLOv11l (réf.)'] for k in labels_x]

axes[1].bar([i - w/2 for i in x], vals_rt,   w, label='RT-DETR-X',      color='#2E86AB', edgecolor='white')
axes[1].bar([i + w/2 for i in x], vals_yolo,  w, label='YOLOv11l (réf.)', color='#E8C547', edgecolor='white')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels_x, fontsize=9)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Comparaison RT-DETR-X vs YOLOv11l', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/rtdetr_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé : /kaggle/working/rtdetr_metrics.png')

# --- Copier les courbes d'entraînement générées par Ultralytics ---
results_dir = '/kaggle/working/runs/detect/rtdetr_autoroute/'
for fname in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    src = os.path.join(results_dir, fname)
    dst = f'/kaggle/working/rtdetr_{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copié : {fname}')
    else:
        print(f'Non trouvé (normal si early stopping): {fname}')

# --- Sauvegarder le meilleur modèle ---
best_pt = os.path.join(results_dir, 'weights/best.pt')
if os.path.exists(best_pt):
    shutil.copy(best_pt, '/kaggle/working/best_model_rtdetr_autoroute.pt')
    size_mb = os.path.getsize('/kaggle/working/best_model_rtdetr_autoroute.pt') / 1024**2
    print(f'\nModèle sauvegardé : /kaggle/working/best_model_rtdetr_autoroute.pt  ({size_mb:.1f} MB)')
else:
    print(f'\n❌ Modèle best.pt introuvable dans {results_dir}')
    print('   Vérifier que l\'entraînement s\'est bien terminé.')

print(f'\n=== Résumé final ===')
print(f'mAP@0.5      : {map50:.4f}')
print(f'mAP@0.5:0.95 : {map5095:.4f}')
print(f'Précision    : {prec:.4f}')
print(f'Rappel       : {rec:.4f}')
print()
print('Pour télécharger : panneau Output à droite → best_model_rtdetr_autoroute.pt → Download')

In [ ]:
# ============================================================
# CELLULE 7 (BONUS) — Inférence de validation & visualisation
# ============================================================
# Cette cellule charge le meilleur modèle entraîné et effectue
# une inférence sur quelques images de validation pour vérifier
# visuellement la qualité des détections.

import random
import os
from pathlib import Path
from ultralytics import RTDETR
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

MODEL_PATH  = '/kaggle/working/best_model_rtdetr_autoroute.pt'
VAL_IMG_DIR = '/kaggle/working/dataset/images/val'
OUT_DIR     = '/kaggle/working/inference_samples/'
os.makedirs(OUT_DIR, exist_ok=True)

# Charger le modèle
model = RTDETR(MODEL_PATH)
print(f'Modèle chargé : {MODEL_PATH}')

# Sélectionner 6 images aléatoires de validation
all_imgs = sorted([f for f in os.listdir(VAL_IMG_DIR) if f.endswith('.jpg')])
samples  = random.sample(all_imgs, min(6, len(all_imgs)))

print(f'Inférence sur {len(samples)} images de validation...')

for img_name in samples:
    img_path = os.path.join(VAL_IMG_DIR, img_name)
    results  = model.predict(source=img_path, conf=0.25, iou=0.5, save=True,
                             project=OUT_DIR, name='predict', exist_ok=True)
    n_det = sum(len(r.boxes) for r in results)
    print(f'  {img_name} — {n_det} détection(s)')

# Afficher les résultats dans une grille
predict_dir = os.path.join(OUT_DIR, 'predict')
images_out  = sorted([f for f in os.listdir(predict_dir) if f.endswith('.jpg')])[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('RT-DETR-X — Exemples de détection (val)', fontsize=14, fontweight='bold')

for ax, fname in zip(axes.flatten(), images_out):
    img = mpimg.imread(os.path.join(predict_dir, fname))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(fname, fontsize=8)

# Masquer les axes vides si moins de 6 images
for ax in axes.flatten()[len(images_out):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/rtdetr_inference_samples.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grille de détection sauvegardée : /kaggle/working/rtdetr_inference_samples.png')